# 01 · Why Frameworks?

**Module 06 · Agent Frameworks**

So far we built everything by hand: the tool descriptions, the agent loop, the router. That was on purpose. You now know exactly what is inside an agent.

In this notebook we do two things:

1. **Look back** at what we wrote by hand, and count how much of it every agent project repeats.
2. **Look at the map** of agent frameworks: the kinds that exist, what each adds, and what each costs.

Then notebooks 02 to 04 build the *same* support desk in three popular frameworks, and notebook 05 compares them.

> 📁 The shop's tools are in `shop_tools.py`. The hand-built desk is in `plain_desk.py`. Use the module's own environment from `requirements.txt` (see the concepts page).

In [1]:
import json
import shop_tools as shop
from shop_tools import client
from pydantic import BaseModel

print("Ready")

Ready


## The three jobs we did by hand

Every agent project needs the same three things.

**Job 1: describe each tool to the model.** The model cannot read our Python. We write a JSON description for every tool.

In [2]:
def make_tool(name, description, input_name, input_description):
    """Build the description of a tool that takes a single text input."""
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": {
                "type": "object",
                "properties": {input_name: {"type": "string", "description": input_description}},
                "required": [input_name],
            },
        },
    }


LOOKUP_ORDER_TOOL = make_tool("lookup_order", "Get the details of one order: item, price, status and delivery date.",
                              "order_id", "The order id, for example A1001")
SEARCH_POLICIES_TOOL = make_tool("search_policies", "Search the shop's policies (refunds, shipping, warranty, payments).",
                                 "question", "What to look up, in plain words")
CHECK_REFUND_TOOL = make_tool("check_refund_eligibility", "Check whether an order can be refunded under the shop's policy.",
                              "order_id", "The order id, for example A1001")
ISSUE_REFUND_TOOL = make_tool("issue_refund", "Issue a refund for an order. Only use this after the order is confirmed eligible.",
                              "order_id", "The order id, for example A1001")

Four tools, and each one needed its own description that repeats what the function already says.

**Job 2: the agent loop.** Ask the model. If it wants a tool, run it, give back the result, and ask again.

In [3]:
def run_agent(agent, message, max_steps=6):
    """Give one message to an agent. It may use its tools several times, then it replies.
    Returns the reply and the tokens used."""
    messages = [
        {"role": "system", "content": agent["instructions"]},
        {"role": "user", "content": message},
    ]
    tokens = 0

    for step in range(max_steps):
        response = client.chat.completions.create(model=shop.MODEL, messages=messages, tools=agent["tools"])
        reply = response.choices[0].message
        messages.append(reply)
        tokens = tokens + response.usage.total_tokens

        # No tool requested? The agent has its answer.
        if not reply.tool_calls:
            return reply.content, tokens

        # Otherwise run each requested tool and give the result back to the agent
        for tool_call in reply.tool_calls:
            args = json.loads(tool_call.function.arguments)
            result = agent["functions"][tool_call.function.name](**args)
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})

    return "Stopped: too many steps.", tokens

**Job 3: the router.** A small model call that decides which specialist should handle an email.

In [4]:
class Route(BaseModel):
    specialist: str  # "orders", "policy" or "refunds"


def choose_specialist(email):
    """Ask the model which specialist should handle this email. Returns the name and the tokens used."""
    response = client.chat.completions.parse(
        model=shop.MODEL,
        messages=[
            {"role": "system", "content": "Pick the specialist for this customer email. "
                                          "orders = where an order is or what is in it. "
                                          "policy = shop rules such as shipping, warranty or payments. "
                                          "refunds = the customer wants their money back."},
            {"role": "user", "content": email},
        ],
        response_format=Route,
    )
    return response.choices[0].message.parsed.specialist, response.usage.total_tokens

Now we put them together and run the desk by hand.

In [5]:
# An agent is a name, instructions, tools, and the functions behind the tools
orders_agent = {"name": "Orders agent", "instructions": shop.ORDERS_INSTRUCTIONS,
                "tools": [LOOKUP_ORDER_TOOL], "functions": {"lookup_order": shop.lookup_order}}

email = "Hi, can you tell me where my order A1002 is? Thanks!"

specialist, router_tokens = choose_specialist(email)
print("Router picked:", specialist)

reply, agent_tokens = run_agent(orders_agent, email)
print("Reply:", reply)
print("Tokens used:", router_tokens + agent_tokens)

Router picked: orders


Reply: Your order A1002, which includes running shoes priced at $120, is currently in transit. It is expected to be delivered by November 25, 2026.
Tokens used: 445


It works. But look at how much of that code has **nothing to do with a support desk**. It is plumbing, and every agent project needs the same plumbing.

## How much plumbing was it?
Let us count the lines of each job in `plain_desk.py`. (Blank lines and comments are not counted.)

In [6]:
def count_block_lines(file_name, block_name):
    """Count the lines of code between '# --- name start ---' and '# --- name end ---' in a file."""
    inside = False
    count = 0
    with open(file_name, encoding="utf-8") as file:
        for line in file:
            if "# --- " + block_name + " end ---" in line:
                inside = False
            if inside and line.strip() != "" and not line.strip().startswith("#"):
                count = count + 1
            if "# --- " + block_name + " start ---" in line:
                inside = True
    return count


total = 0
for job in ["tools", "loop", "router"]:
    lines = count_block_lines("plain_desk.py", job)
    total = total + lines
    print(job.ljust(8) + str(lines) + " lines")
print("-" * 20)
print("total".ljust(8) + str(total) + " lines of plumbing")

tools   22 lines
loop    20 lines
router  16 lines
--------------------
total   58 lines of plumbing


And this is a **small** desk: no memory, no guardrails, no human approval, no logging, no tracing. Each of those adds more plumbing.

## What a framework gives you
A framework is a package of that plumbing, written once and tested by many people.

| You write by hand | A framework gives you |
|---|---|
| The JSON description of each tool | Built **from your function** (its name, docstring and type hints) |
| The agent loop | A ready-made runner, with step limits |
| The router or handoff | Handoffs, graphs or crews |
| Keeping state between steps | State objects and **checkpoints** (save and resume) |
| Pausing for a human | Built-in pause and resume |
| Printing to see what happened | **Tracing** and event logs |
| Streaming, retries, async | Ready to use |
| Guardrails (Module 05) | Hooks where checks can be attached |

## What a framework costs
| Cost | What it means |
|---|---|
| **A new vocabulary** | Agents, runners, nodes, crews, tasks. Each framework names things its own way. |
| **Hidden prompts and extra model calls** | The framework writes prompts for you, so you may spend more tokens than you expect. We will measure this. |
| **Churn** | Agent frameworks change fast. Some are still below version 1.0 and break between releases. That is why we **pin versions**. |
| **Harder debugging** | When something goes wrong, the bug may be inside the framework, in code you did not write. |
| **Lock-in** | Your agents and flows are written in one framework's style. Moving later takes work. |

## The map of frameworks
There are many. They fall into a few kinds, depending on **how they think about an agent system**.

| Kind | The idea | Examples | Good for |
|---|---|---|---|
| **Lightweight agent SDKs** | A few building blocks: agent, tool, handoff | **OpenAI Agents SDK**, Pydantic AI, Claude Agent SDK, Google ADK | Getting an agent running quickly with little new vocabulary |
| **Graph / workflow engines** | You draw the flow: boxes and arrows with shared state | **LangGraph** | Flows that need control, loops, saving progress, and human approval |
| **Role-based teams** | Agents with a role, a goal and a backstory, working through tasks | **CrewAI** | Work that reads like a team doing a job |
| **Conversation-based** | Agents talk to each other in a chat to solve a problem | AutoGen (now in maintenance mode; Microsoft points new projects to **Microsoft Agent Framework**) | Research and experiments |
| **Low-code / visual** | Drag and drop boxes on a canvas | n8n, Flowise, Dify | Teams that do not want to write code |
| **Data-first** | Built around your documents and retrieval | LlamaIndex | RAG-heavy applications |

We **study three in this module**: the OpenAI Agents SDK, LangGraph and CrewAI. They are among the most popular, and each one represents a different way of thinking: *handoffs*, *graphs* and *crews*.

> The names on this map change often. Do not memorize products. Learn the **ideas** (handoff, graph, crew), because every new framework is a mix of them.

## When you do NOT need a framework
- **One agent with a few tools.** The loop in this notebook is about 20 lines. A framework adds more than it removes.
- **You need to understand every token.** For a cost-critical or safety-critical path, hand-written code is easier to inspect.
- **A workflow with no AI decisions.** If the steps never change, plain code is simpler and cheaper.

Reach for a framework when you have **several agents, loops, saved state, human approval or a team that must maintain the code**.

Next: the same support desk, built with the **OpenAI Agents SDK**.